# B2-021 — Session 5: Vision-System Design and Audit

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.


## 1. Choose the prediction object before the architecture

Classification returns one label per image or graph; detection returns a variable set of boxes/classes; segmentation returns one class per pixel; graph framing predicts from nodes/edges. Begin with the deployment question and target structure. Do not choose a fashionable backbone first and force the target into its output.

**Checkpoint 1A.** Which framing preserves pixel boundaries?

**Checkpoint 1B.** Which framing returns locations and categories?

**Collected answers.** Segmentation; detection.


## 2. Match heads, losses, and metrics

Classification uses `(B,C)` logits, cross-entropy, and accuracy or macro-F1. Detection uses objectness/class/box heads, BCE/CE/Smooth-L1, and IoU plus precision/recall. Segmentation uses `(B,C,H,W)`, pixel CE, and Dice/IoU. Graph classification uses graph-token logits, CE, and held-out accuracy. A training loss need not be the reporting metric.

**Checkpoint 2A.** Which metric measures box overlap?

**Checkpoint 2B.** Why not report pixel accuracy alone?

**Collected answers.** IoU; background dominance can hide failed foreground segmentation.


## 3. Compare CNN and ViT constraints

With small data, strict latency, and local textures, a compact CNN is usually the safer baseline. With enough data/compute and a need for flexible global token interaction, a ViT may be appropriate. Compare patch count, quadratic attention score size, parameter count, and known inductive bias. Neither architecture earns credit without a held-out protocol.

**Checkpoint 3A.** What makes a ViT expensive at high resolution?

**Checkpoint 3B.** Which baseline is data-efficient by design?

**Collected answers.** Quadratic token-pair scores; a CNN's locality bias often improves small-data efficiency.


## 4. Split and feature-identity audit

Split by independent example or group before training. Require train and held-out IDs disjoint, resolve each actual structured feature through its canonical fingerprint, and separately verify the aligned target fingerprint. Repeated labels are allowed; duplicate feature fingerprints are not. Unknown features, target substitution, and one-row-in-both-splits are hard failures.

**Checkpoint 4A.** Why is target fingerprint uniqueness wrong?

**Checkpoint 4B.** What must be unique instead?

**Collected answers.** Many examples legitimately share a class; the structured feature fingerprint must uniquely identify an example.


## 5. End-to-end architecture review

Review in this order: task target → literal features → shape/dtype ledger → representation → output head → loss masking/reduction → optimizer protocol → immutable split → held-out metric → error table. At each arrow name the owning module and whether gradients flow. For cross-modal attention, explicitly name Q source and K/V source.

**Checkpoint 5A.** Where is softmax omitted?

**Checkpoint 5B.** What does a shape ledger catch?

**Collected answers.** Omit it before CE/BCE-with-logits; the ledger catches axis, channel, token, and resolution mismatches.


## 6. Failure analysis across four systems

For ViT, separate patch-order errors from data scarcity. For detection, separate false positives, wrong class, and duplicate boxes. For U-Net, separate spatial misalignment from leakage and background-only predictions. For graphs, separate invalid edges, aggregation-axis errors, split leakage, and Q/K/V reversal. Diagnose the earliest violated contract, not only the last bad metric.

**Checkpoint 6A.** What typically causes duplicate boxes?

**Checkpoint 6B.** What trace reveals segmentation leakage?

**Collected answers.** Missing/incorrect NMS; held-out IDs or masks entering train batches, threshold selection, or augmentation fitting.


## 7. Worked laboratory: end-to-end design and leakage audit

Suppose the deployment question is: given an 8x8 grayscale image, return a foreground mask and summarize it with a graph of four regions. Begin with two prediction objects, not one vague “vision model”: pixel labels `(B,8,8)` and a graph label `(B,)`. The segmentation branch uses the pinned one-down U-Net because boundary alignment matters. The graph branch receives declared node measurements and directed adjacency, mean-aggregates neighbors, then uses a class token. If graph facts later refine image tokens, image owns Q and graph owns K/V.

Write the complete ledger. Images are float32 `(B,1,8,8)`. U-Net logits are `(B,2,8,8)` and pixel targets int64 `(B,8,8)`. Graph node features are float32 `(B,4,3)`, adjacency uint8 `(B,4,4)`, aggregates `(B,4,3)`, attention input `(B,5,8)`, and class logits `(B,2)`. If cross-modal attention is added, choose unequal token counts during testing and assert scores `(B,N_image,N_graph)`. Every arrow must name its module and whether targets are available there.

Now define losses and metrics separately. Pixel CE trains segmentation; hard foreground Dice and IoU report overlap. Graph CE trains class logits; held-out correct count reports the tiny certificate. If a detection branch were selected instead, independent objectness, class, and box heads would use BCE, CE, and Smooth-L1, while decoded box IoU and class accuracy would report behavior. Never claim that a lower differentiable loss guarantees the discrete metric.

Split before fitting anything. Store exact ordered train and held-out ID tuples per task. The batch API accepts only one entire tuple in stored order; duplicate, reversed, partial, unknown, and opposite-task sequences fail. Then resolve every actual feature row by a canonical fingerprint that includes field name, array type, dtype, shape, and contiguous bytes. Verify the corresponding target fingerprint separately. This prevents a trusted ID string from blessing substituted bytes.

The first code cell is a compact split audit. Extend its logic conceptually to groups such as patient, video, or source document: related samples belong wholly to one split. The held-out split may be evaluated at fixed checkpoints, but it may not choose normalization statistics, thresholds, augmentations, early stopping, model width, or graph edges. If any of those decisions uses held-out values, the final metric is no longer an unbiased certificate.

Perform an architecture audit in causal order: target object; raw literal feature; transform; shape and dtype; representation; head; loss mask and reduction; optimizer protocol; held-out metric; error table. For each failure, name the earliest violated invariant. A 7x8 decoder is first an alignment error, not a poor-Dice problem. A reversed Q/K/V trace is first a semantic-direction error, not merely a strange attention map. A target-derived edge is leakage even if validation accuracy looks plausible.

The second code cell encodes a miniature audit record and checks that image queries, graph keys/values, score shape, and split disjointness agree. A useful audit report includes the offending evidence, affected semantics, and first exact repair. “Be careful” is not a repair; “remove held-out ID from optimizer tuple and rebuild the full stored-order training batch” is.

Finish with falsification. For the U-Net, run a no-skip ablation without changing the held-out selection process. For the graph model, compare directed `A@X` with an intentional transpose mutant. For the ViT, mutate position addition or residual/norm order. For detection, remove the class head. These mutants should fail a specific witness. Passing only the intended implementation is stronger evidence than a green sample that many wrong programs can reproduce.

Write the final audit as a decision record. State the chosen task framing, rejected alternatives, tensor contracts, provenance boundary, train-only fitting operations, immutable held-out role, and one failure-triggered fallback. Include a compact error table with columns for example ID, expected structure, predicted structure, earliest violated invariant, and proposed repair. A repair must preserve the evaluation boundary: for example, changing a decoder stride may be retrained on the original train split, but the held-out masks cannot decide which stride to try. This record makes later review reproducible and prevents a good metric from erasing a broken process.

**Checkpoint 7A.** Name the ordered audit stages from target choice through error analysis.

**Checkpoint 7B.** Which held-out-dependent choices are leakage even if no held-out row enters `backward()`?

**Collected answers.** Audit target, literal features, shapes/dtypes, representation, head, loss, optimizer, immutable split, held-out metric, then errors. Held-out normalization, thresholds, augmentation, early stopping, model selection, or graph construction all leak evaluation information.


In [ ]:
train_ids = ('example-train-00', 'example-train-01', 'example-train-02')
heldout_ids = ('example-heldout-00', 'example-heldout-01')
assert len(train_ids) == len(set(train_ids))
assert len(heldout_ids) == len(set(heldout_ids))
assert set(train_ids).isdisjoint(heldout_ids)

def exact_split(candidate, expected):
    if tuple(candidate) != expected:
        raise ValueError('must equal complete immutable split in stored order')
    return expected

assert exact_split(train_ids, train_ids) == train_ids


In [ ]:
audit = {
    'task': 'update image tokens from graph facts',
    'q_source': 'image_tokens',
    'k_source': 'graph_tokens',
    'v_source': 'graph_tokens',
    'score_shape': ('B', 'N_image', 'N_graph'),
    'optimizer_ids': ('graph-train-00', 'graph-train-01'),
    'heldout_ids': ('graph-heldout-00', 'graph-heldout-01'),
}
assert audit['q_source'] == 'image_tokens'
assert audit['k_source'] == audit['v_source'] == 'graph_tokens'
assert set(audit['optimizer_ids']).isdisjoint(audit['heldout_ids'])
print('audit passes:', audit)


## 8. Common pitfalls and Exam connections

Common pitfalls are comparing unlike metrics, using held-out data to choose architecture, treating example IDs as proof of actual data identity, importing an undeclared pretrained component, and changing several contracts at once. Exam connections reward a concise audit: quote the offending row or tensor, name the violated invariant, and give the first downstream consequence.

**Checkpoint 8A.** Which practice chooses ViT versus CNN?

**Checkpoint 8B.** Which two challenges audit leakage?

**Collected answers.** p21; p23 and p24.


## 9. Forward-only deeper topic: open-world multimodal systems

Forward-only deeper topic: production systems may combine region proposals, text encoders, retrieval, instruction tuning, calibration, and distribution-shift monitoring. Those components require new data/provenance and are not assessed here. B2-021 remains a closed, literal, CPU-only environment where every feature, target, edge, and split is inspectable.

This final section is forward-only and not assessed.

**Checkpoint 9A.** May external text-image data be used?

**Checkpoint 9B.** What is the boundary of this unit?

**Collected answers.** No; original literal fixture data and four auditable task framings only.
